<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I prioritize content that already has search visibility but may have room for improvement. Pages that receive a high number of impressions but have weaker click engagement may represent opportunities to improve their performance

The reason codes this rule can output are:
- HIGH_VISIBILITY_REFRESH: The page receives a high amount of impressions, making it a strong candidate for review
- LOW_CTR_OPPORTUNITY: The page receives impressions but has relatively low click-through performance
- REVIEW: The page shows some potential but does not strongly match the main opportunity signals



In [30]:
# Load dataset
import pandas as pd
from pathlib import Path

file_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not file_path.exists():
    raise FileNotFoundError(f"Can't find {file_path.resolve()}. Check your working directory.")

df = pd.read_csv(file_path)

FileNotFoundError: Can't find /data/raw/content_refresh_anonymized.csv. Check your working directory.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

To create the ranked queue, I combine the signals from my rule into a simple score. The goal is not to perfectly predict which page should be changed, but to create a transparent priority list of pages that may be worth reviewing.

Pages receive a higher score when they:
have stronger search visibility through impressions,
show weaker click engagement compared with their visibility and have more potential impact if improved.


In [16]:
# Calculate CTR from clicks and impressions
df["ctr"] = (
    df["clicks_90d"] /
    df["impressions_90d"]
)
df["ctr"] = df["ctr"].fillna(0)
df[["impressions_90d", "clicks_90d", "ctr"]].head()

,impressions_90d,clicks_90d,ctr
0,3803,29,0.007626
1,15320,7,0.000457
2,12581,11,0.000874
3,11751,58,0.004936
4,19140,24,0.001254


In [19]:
# Normalize visibility
df["visibility_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)
# Lower CTR with visibility creates more opportunity
df["ctr_opportunity"] = (
    1 - (df["ctr"] / df["ctr"].max())
)
df[["visibility_score", "ctr_opportunity"]].head()

,visibility_score,ctr_opportunity
0,0.007346,0.992374
1,0.029592,0.999543
2,0.024301,0.999126
3,0.022698,0.995064
4,0.036970,0.998746


In [27]:
# Final Score
df["score"] = (
    0.6 * df["visibility_score"]
    +
    0.4 * df["ctr_opportunity"]
)


# Reason codes
high_visibility_threshold = df["impressions_90d"].quantile(0.75)
low_ctr_threshold = df["ctr"].median()


def assign_reason(row):
    if row["impressions_90d"] >= high_visibility_threshold:
        return "HIGH_VISIBILITY_REFRESH"
    elif row["ctr"] <= low_ctr_threshold:
        return "LOW_CTR_OPPORTUNITY"
    else:
        return "REVIEW"


df["reason_code"] = df.apply(assign_reason, axis=1)


# Action labels
def assign_action(reason):
    if reason == "HIGH_VISIBILITY_REFRESH":
        return "REVIEW_UPDATE"
    elif reason == "LOW_CTR_OPPORTUNITY":
        return "OPTIMIZE_SNIPPET"
    else:
        return "MONITOR"


df["action"] = df["reason_code"].apply(assign_action)


# Rank queue
baseline_queue = (
    df[
        [
            "content_id",
            "score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
)


baseline_queue.head(10)


# Save CSV
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

OSError: Cannot save file into a non-existent directory: 'work/outputs'

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
top20 = baseline_queue.head(20)

top20

NameError: name 'baseline_queue' is not defined

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be weaker because the rule only uses a few performance signals. A page may receive many impressions but still not be the best candidate for improvement.

### Leakage check
I confirmed that the score only uses historical performance features:
- impressions_90d
- clicks_90d
- calculated CTR

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.